# 04 · Modeling

**Notebook:** 04 of 04  

**Goal:** Build preprocessing pipelines, compare models, and finalize a production-ready final pipeline with best model.

### What this notebook covers
- Numeric pipeline: KNN imputation + StandardScaler
- Categorical pipeline: most-frequent imputation + OneHotEncoder
- Baseline test with Logistic Regression (pipeline smoke test)
- Multi-model comparison with SMOTE-Tomek (LR, SVM, RF, XGB, KNN)
- Final SVM pipeline — best balance of accuracy and recall
- Save final model 

In [51]:
import plotly.express as px
import pandas as pd
import numpy as np

In [52]:
# ── Load final features from feature engineering notebook  ─────────────────────────────────────
df= pd.read_csv("/Users/mohammedmahmood/Desktop/Fresco/files/data/03_features_model.csv")
df_streamlit= pd.read_csv("/Users/mohammedmahmood/Desktop/Fresco/files/data/03_features_deploy.csv")

# **Modeling**

In [53]:
from sklearn.preprocessing import OneHotEncoder , StandardScaler , LabelEncoder , RobustScaler
from sklearn.impute import SimpleImputer , KNNImputer
from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB , GaussianNB
from sklearn.metrics import accuracy_score , recall_score , precision_score , f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split , cross_validate, GridSearchCV, KFold 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import make_scorer, recall_score, accuracy_score, precision_score, f1_score

In [54]:
# split data into x and y
x = df.drop("Loan_Status" , axis = 1 )
y = df["Loan_Status"]

### **1. Check number of categorical nulls, and number of unique value to decide what encoding technique we will use**

In [55]:
Cat_Columns = df.select_dtypes(include = "object")
Cat_Columns.isna().sum()

Married           3
Dependents       15
Property_Area     0
dtype: int64

In [56]:
# print all number of unique values in Cat_Columns
for col in Cat_Columns:
    print(col, df[col].nunique())

Married 2
Dependents 4
Property_Area 3


### **2- Check number of numerical null values to handle by Ml pipeline**

In [57]:
Num_Columns = df.select_dtypes(include="number")
Num_Columns.isna().sum() 

Loan_Amount_Term           14
Credit_History             50
Loan_Status                 0
log_ApplicantIncome         0
log_LoanAmount             22
log_Total_Income            0
log_Loan_Monthly_Paid      36
log_Income_After_Loan      41
log_Income_to_LoanRatio    22
dtype: int64

# 3- Pipeline creation 

###  created 2 pipelines to handle features separately:
#### - One for categorical features (encoding, imputing missing)
#### - One pipeline for numerical features (scaling, imputation)

In [58]:
# created nummeric and categorical pipline to handle every one
Num_Columns = x.select_dtypes(include="number").columns
Cat_Columns = x.select_dtypes(include = "object_").columns

#### **1- Numeric Pipeline**

In [59]:
# Nummeric pipline
Num_Steps = [
    ("Num_Imputer", KNNImputer()),
    ("Scaler", StandardScaler())
]

Num_Pipeline = Pipeline(steps=Num_Steps)

In [60]:
Num_Pipeline 

Pipeline(steps=[('Num_Imputer', KNNImputer()), ('Scaler', StandardScaler())])

#### **2- Categorical Pipeline**

In [61]:
Cat_Steps = [
    ("Cat_Imputer", SimpleImputer(strategy="most_frequent")),
    ("Encoder", OneHotEncoder(sparse_output=False, drop="first"))
]

Cat_Pipeline = Pipeline(steps=Cat_Steps)

In [62]:
Cat_Pipeline

Pipeline(steps=[('Cat_Imputer', SimpleImputer(strategy='most_frequent')),
                ('Encoder', OneHotEncoder(drop='first', sparse_output=False))])

In [63]:
Cat_Columns.isna().sum()

0

# 3. ColumnTransformer 

In [64]:
'''
Create a ColumnTransformer that applies:
- Numeric Pipeline to all numeric columns
- Categorical Pipeline to all categorical columns
Any remaining columns will be passed through unchanged
'''

Transformer = ColumnTransformer(
    transformers= [
    ("Num", Num_Pipeline, Num_Columns) ,
    ("Cat", Cat_Pipeline, Cat_Columns) ]
     , remainder="passthrough"
     )


## **4-Pipeline Testing**

- #### A base model will be trained using the preprocessing pipeline we created.  
- #### This test ensures that the pipeline is functioning properly before we evaluate multiple models.

In [65]:
steps = [

    ("Preprocessing" , Transformer),
    ("Model" , LogisticRegression()), # Initial model to just test pipline
]

pipeline = Pipeline(steps = steps)

# Cross-validation with to evaluate models
scores = cross_validate(pipeline, x, y, cv = 5, scoring="accuracy" ,return_train_score=True)

In [66]:
# Train & Test  results
print("training results:", scores["train_score"].mean())
print("Testing results:", scores["test_score"].mean())

training results: 0.8075073776964962
Testing results: 0.8043182726909237


### **pipline is working ,  Now try all models and take the best one in performance**

In [67]:
# Check number of value in each class in target to aplly SMOTE Tomek
y.value_counts()


Loan_Status
1    422
0    191
Name: count, dtype: int64

# **5- SMOTE-Tomek Optimization**

##### tried several values with SMOTE-Tome choose the best sampling ratio and integrate SMOTE into the final ML preprocessing pipeline

In [68]:
'''
applied SMOTE to balance the minority class. After applying SMOTE, the code displays the number of instances in both classes.  

will use these results to choose the best sampling ratio and integrate SMOTE into the final ML preprocessing pipeline.

tried several sampling ratio values with SMOTE-Tomek and found that using **300 samples for the minority class** provides the best results. 
'''

# Make sure X and y are aligned
X = df.drop("Loan_Status", axis=1)
y = df["Loan_Status"]

# Fit-transform
X_transformed = Transformer.fit_transform(X)

# Convert to DataFrame to preserve feature names
feature_names = Transformer.get_feature_names_out()
feature_names = [col.split("__")[-1] for col in feature_names]  # clean names

X_transformed = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)

# Align y
y = y.loc[X_transformed.index]

from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from collections import Counter

print("Before:", Counter(y))

smote = SMOTETomek(smote=SMOTE(sampling_strategy={0: 300}, random_state=24))
X_res, y_res = smote.fit_resample(X_transformed, y)

print("After:", Counter(y_res))

Before: Counter({1: 422, 0: 191})
After: Counter({1: 393, 0: 271})


## **6- Model Evaluation and Selection**

#### Several machine learning models will be trained and evaluated, The best performing model will then be selected, tuned and deployed

In [69]:
# Models will used
models = [
    ("LR" , LogisticRegression()) ,
    ("SVM" , SVC(gamma= .09)) ,
    ("CART" , DecisionTreeClassifier()) ,
    ("RF" , RandomForestClassifier()) ,
    ("RF" , RandomForestClassifier()),
    ("XG" , XGBClassifier())
]

In [70]:
"""
This block of code evaluates multiple machine learning models using a pipeline approach that apply:

1- Preprocessing : applies transformations we do

2- SMOTETomek : balances the dataset by oversampling the minority class

3- Cross-validation: ensures fair evaluation

4- Scoring metrics : measures accuracy, recall, precision, for both training and test sets.

5- Results : prints average scores so we can compare model performance and detect overfitting.

"""

# scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score),
    "f1": make_scorer(f1_score)
}

for model in models:

    steps = [
        ("Preprocessing", Transformer),
        ("SmoteTomek", SMOTETomek(smote=SMOTE(sampling_strategy={0: 300}, random_state=24))), # using Smote Tomic to handle Imbalanced class
        (model )
        ]

    # pipeline contain all preprocessing steps we do
    Final_pipeline = Pipeline(steps=steps)

    # Cross-validation with shuffle
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=24)
    scores = cross_validate(Final_pipeline, x, y, scoring= scoring, cv=kf, return_train_score=True)

    # Print results of Recall and Accuracy
    print(model[0])
    print("Train Accuracy:", round(scores["train_accuracy"].mean(), 3))
    print("Test Accuracy:", round(scores["test_accuracy"].mean(), 3))
    print("Train Recall:", round(scores["train_recall"].mean(), 3))
    print("Test Recall:", round(scores["test_recall"].mean(), 3))
    print("Train Precision:", round(scores["train_precision"].mean(), 3))
    print("Test Precision:", round(scores["test_precision"].mean(), 3))
    print("Train f1:", round(scores["train_f1"].mean(), 3))
    print("Test f1:", round(scores["test_f1"].mean(), 3))
    print("-" * 40)
    print("\n")


LR
Train Accuracy: 0.78
Test Accuracy: 0.767
Train Recall: 0.867
Test Recall: 0.86
Train Precision: 0.823
Test Precision: 0.813
Train f1: 0.844
Test f1: 0.835
----------------------------------------


SVM
Train Accuracy: 0.843
Test Accuracy: 0.803
Train Recall: 0.951
Test Recall: 0.936
Train Precision: 0.842
Test Precision: 0.808
Train f1: 0.893
Test f1: 0.867
----------------------------------------


CART
Train Accuracy: 0.97
Test Accuracy: 0.724
Train Recall: 0.986
Test Recall: 0.818
Train Precision: 0.97
Test Precision: 0.789
Train f1: 0.978
Test f1: 0.803
----------------------------------------


RF
Train Accuracy: 0.971
Test Accuracy: 0.788
Train Recall: 0.993
Test Recall: 0.903
Train Precision: 0.965
Test Precision: 0.811
Train f1: 0.979
Test f1: 0.854
----------------------------------------


RF
Train Accuracy: 0.971
Test Accuracy: 0.783
Train Recall: 0.992
Test Recall: 0.891
Train Precision: 0.967
Test Precision: 0.813
Train f1: 0.979
Test f1: 0.85
-------------------------

## **Choosing Svm as a best model**

- Train Accuracy: 0.841
- Test Accuracy: 0.807
- Train Recall: 0.973
- Test Recall: 0.965
- Train f1: 0.889
- Test f1: 0.841

### **Tuning was not done because the dataset was small, and earlier tries at tuning gave bad results most of the time.**

In [71]:
# Final Pipline with Chosen model SVM

steps = [
    ("Preprocessing" , Transformer) ,
    ("SmoteTomek", SMOTETomek(smote=SMOTE(sampling_strategy={0: 300}, random_state=24))),
    ("SVM" , SVC(gamma= .09))
]

Final_pipeline = Pipeline(steps = steps)


### **final model with all things (all preprocessing + Best model) , ready to production**

In [72]:
Final_model = Final_pipeline.fit(x,y) 

In [73]:
Final_model

Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Num',
                                                  Pipeline(steps=[('Num_Imputer',
                                                                   KNNImputer()),
                                                                  ('Scaler',
                                                                   StandardScaler())]),
                                                  Index(['Loan_Amount_Term', 'Credit_History', 'log_ApplicantIncome',
       'log_LoanAmount', 'log_Total_Income', 'log_Loan_Monthly_Paid',
       'log_Income_After_Loan', 'log_Income_to_LoanRatio'],
      dtype='object')),
                                                 ('Cat',
                                                  Pipeline(steps=[('Cat_Imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False))]),
                                                  Index(['Married', 'Dependents', 'Property_Area'], dtype='object'))])),
                ('SmoteTomek',
                 SMOTETomek(smote=SMOTE(random_state=24,
                                        sampling_strategy={0: 300}))),
                ('SVM', SVC(gamma=0.09))])

In [74]:
# For deploy
df_streamlit.columns 

Index(['Married', 'Dependents', 'ApplicantIncome', 'CoapplicantIncome',
       'LoanAmount', 'Credit_History', 'Property_Area', 'Loan_Status',
       'Total_Income', 'Income_After_Loan', 'Income_to_LoanRatio'],
      dtype='object')

In [75]:
# import joblib

# joblib.dump(Final_model, "Final_model_SVM.joblib")

In [76]:
import sklearn

print(sklearn.__version__)

1.3.2
